In [1]:
import kagglehub

# Capture the root path where the Kaggle dataset is downloaded
DATASET_ROOT = kagglehub.dataset_download('banuprasadb/visdrone-dataset')

print('Data source import complete.')
print(f'Dataset downloaded to: {DATASET_ROOT}')

100%|██████████| 2.10G/2.10G [01:41<00:00, 22.2MB/s]

Extracting files...


Data source import complete.
Dataset downloaded to: /root/.cache/kagglehub/datasets/banuprasadb/visdrone-dataset/versions/1


In [2]:
!git clone https://github.com/ultralytics/ultralytics -b main
%pip install -qe ultralytics

Cloning into 'ultralytics'...
remote: Enumerating objects: 84687, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 84687 (delta 59), reused 35 (delta 34), pack-reused 84598 (from 2)
Receiving objects: 100% (84687/84687), 45.15 MiB | 16.50 MiB/s, done.
Resolving deltas: 100% (63939/63939), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done


In [3]:
import yaml
import os

# Construct the base path for the VisDrone dataset using the downloaded root
# Assuming VisDrone_Dataset is directly inside the DATASET_ROOT
visdrone_base_path = os.path.join(DATASET_ROOT, 'VisDrone_Dataset')

# Define your YAML content as a multi-line string
yaml_content = f"""
# Dataset paths
path: {visdrone_base_path}  # dataset root dir
train: {os.path.join(visdrone_base_path, 'VisDrone2019-DET-train', 'images')}  # train images
val: {os.path.join(visdrone_base_path, 'VisDrone2019-DET-val', 'images')}    # val images
test: {os.path.join(visdrone_base_path, 'VisDrone2019-DET-test-dev', 'images')} # test images

#number of classes
nc: 10

# Class names
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

# Write the YAML content to a file in the current working directory (/content/ in Colab)
with open('dataset.yaml', 'w') as f:
    f.write(yaml_content)

In [6]:
!yolo train model=yolo11m.pt data=dataset.yaml epochs=50 imgsz=640 patience=35 cache=true batch=16 save=true device=0 project=YOLO11m_VisDrone name=run optimizer=SGD

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=35, perspective=0.0, plots=True, pose=12.

In [12]:
import os
import pandas as pd
import matplotlib.pyplot as plt

print("Training completed! Loading results and generating evaluation metrics...")

# Path to results CSV file (assuming YOLO project outputs to the correct directory)
results_csv_path = '/content/ultralytics/runs/detect/YOLO11m_VisDrone/run/results.csv'

# Load training results
results = pd.read_csv(results_csv_path)

# Remove leading/trailing whitespace from column names
results.columns = results.columns.str.strip()
print(results.columns)


# Extract final metrics (last epoch)
final_metrics = results.iloc[-1]
map_50 = final_metrics['metrics/mAP50(B)']
map_50_95 = final_metrics['metrics/mAP50-95(B)']
precision = final_metrics['metrics/precision(B)']
recall = final_metrics['metrics/recall(B)']

# Display nicely
print("\n" + "="*50)
print("FINAL EVALUATION METRICS")
print("="*50)
print(f"mAP@0.5: {map_50:.4f}")
print(f"mAP@0.5:0.95: {map_50_95:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print("="*50)

Training completed! Loading results and generating evaluation metrics...
Index(['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
       'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)',
       'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss',
       'lr/pg0', 'lr/pg1', 'lr/pg2'],
      dtype='object')

FINAL EVALUATION METRICS
mAP@0.5: 0.4627
mAP@0.5:0.95: 0.2835
Precision: 0.5625
Recall: 0.4430


In [ ]:
!zip -r YOLO11m_VisDrone.zip YOLO11m_VisDrone

In [ ]:
!yolo val model=YOLO11m_VisDrone/run/weights/best.pt data=dataset.yaml save=True

# Task
The model training results are saved in the `ultralytics/runs/detect` directory, but the notebook is currently looking for `results.csv` in `YOLO11m_VisDrone/run/results.csv`, which is incorrect and causes a `FileNotFoundError`. To fix this, I'll update `results_csv_path` to point to the correct location of `results.csv` within the `ultralytics` directory. Then, I'll update the path for the `!zip` command to ensure the correct directory is zipped.

After these corrections, I will execute the relevant cells to:
1. Display the evaluation metrics from `results.csv`.
2. Create a zip archive named `YOLO11m_VisDrone.zip` containing all the training artifacts.
3. Provide a way to download this zip file to your local machine.

This will ensure that the evaluation metrics are displayed and all training artifacts are available for download.

## Correct Results Path and Display Metrics

### Subtask:
Adjust the `results_csv_path` in the evaluation cell to correctly point to the `results.csv` file generated by the YOLO training. This file is located within the `ultralytics/runs/detect` directory. After correcting the path, the cell will be executed to display the final evaluation metrics.


**Reasoning**:
The previous attempt to load `results.csv` failed because the `results_csv_path` was incorrect. I need to update the path to the `results.csv` file, which is located in `/content/ultralytics/runs/detect/YOLO11m_VisDrone/run/results.csv` based on the given instructions, and then re-run the code to display the evaluation metrics.



In [7]:
import os
import pandas as pd
import matplotlib.pyplot as plt

print("Training completed! Loading results and generating evaluation metrics...")

# Path to results CSV file (assuming YOLO project outputs to the correct directory)
results_csv_path = '/content/ultralytics/runs/detect/YOLO11m_VisDrone/run/results.csv'

# Load training results
results = pd.read_csv(results_csv_path)

# Remove leading/trailing whitespace from column names
results.columns = results.columns.str.strip()
print(results.columns)


# Extract final metrics (last epoch)
final_metrics = results.iloc[-1]
map_50 = final_metrics['metrics/mAP50(B)']
map_50_95 = final_metrics['metrics/mAP50-95(B)']
precision = final_metrics['metrics/precision(B)']
recall = final_metrics['metrics/recall(B)']

# Display nicely
print("\n" + "="*50)
print("FINAL EVALUATION METRICS")
print("="*50)
print(f"mAP@0.5: {map_50:.4f}")
print(f"mAP@0.5:0.95: {map_50_95:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print("="*50)

Training completed! Loading results and generating evaluation metrics...
Index(['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
       'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)',
       'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss',
       'lr/pg0', 'lr/pg1', 'lr/pg2'],
      dtype='object')

FINAL EVALUATION METRICS
mAP@0.5: 0.4627
mAP@0.5:0.95: 0.2835
Precision: 0.5625
Recall: 0.4430


**Reasoning**:
The previous `!zip` command might not be targeting the correct directory for zipping all training artifacts. Based on the `YOLO11m_VisDrone` project name used in the training cell and the standard Ultralytics output structure, the training artifacts, including `results.csv` and the `weights` directory, are located at `/content/ultralytics/runs/detect/YOLO11m_VisDrone/run`. I need to ensure the correct directory is zipped, and the zip file should be named appropriately. Then, I will add a step to download the generated zip file.



In [9]:
import os
from google.colab import files

# Define the directory to zip
train_output_dir = '/content/ultralytics/runs/detect/YOLO11m_VisDrone/run'
zip_file_name = 'YOLO11m_VisDrone_run_artifacts.zip'

# Zip the directory
!zip -r {zip_file_name} {train_output_dir}

# Provide a link to download the zip file
print(f"\nZipping complete. You can download the training artifacts from {zip_file_name}")
files.download(zip_file_name)

  adding: content/ultralytics/runs/detect/YOLO11m_VisDrone/run/ (stored 0%)
  adding: content/ultralytics/runs/detect/YOLO11m_VisDrone/run/confusion_matrix_normalized.png (deflated 15%)
  adding: content/ultralytics/runs/detect/YOLO11m_VisDrone/run/train_batch16202.jpg (deflated 8%)
  adding: content/ultralytics/runs/detect/YOLO11m_VisDrone/run/train_batch0.jpg (deflated 2%)
  adding: content/ultralytics/runs/detect/YOLO11m_VisDrone/run/val_batch1_pred.jpg (deflated 3%)
  adding: content/ultralytics/runs/detect/YOLO11m_VisDrone/run/weights/ (stored 0%)
  adding: content/ultralytics/runs/detect/YOLO11m_VisDrone/run/weights/last.pt (deflated 8%)
  adding: content/ultralytics/runs/detect/YOLO11m_VisDrone/run/weights/best.pt (deflated 8%)
  adding: content/ultralytics/runs/detect/YOLO11m_VisDrone/run/val_batch0_pred.jpg (deflated 4%)
  adding: content/ultralytics/runs/detect/YOLO11m_VisDrone/run/val_batch1_labels.jpg (deflated 3%)
  adding: content/ultralytics/runs/detect/YOLO11m_VisDrone/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Reasoning**:
The training process is complete, the evaluation metrics have been displayed, and the training artifacts have been zipped and made available for download. The next logical step is to execute the validation command as per the original notebook's flow to get explicit validation results.



In [10]:
!yolo val model=YOLO11m_VisDrone/run/weights/best.pt data=dataset.yaml save=True

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/content/ultralytics/ultralytics/cfg/__init__.py", line 956, in entrypoint
    model = YOLO(model, task=task)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ultralytics/ultralytics/models/yolo/model.py", line 76, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/content/ultralytics/ultralytics/engine/model.py", line 144, in __init__
    self._load(model, task=task)
  File "/content/ultralytics/ultralytics/engine/model.py", line 283, in _load
    self.model, self.ckpt = load_checkpoint(weights)
                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ultralytics/ultralytics/nn/tasks.py", line 1507, in load_checkpoint
    ckpt, weight = torch_safe_load(weight)  # load ckpt
                   ^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ultralytics/ultralytics/nn/tasks.py", line 1455, in torch_safe_

**Reasoning**:
The previous `!yolo val` command failed because the model path was incorrect. The model weights are located at `/content/ultralytics/runs/detect/YOLO11m_VisDrone/run/weights/best.pt`, not relative to the current directory. I need to update the `model` argument in the `!yolo val` command with the absolute path.



In [11]:
import os

# Define the absolute path to the best trained model weights
model_path = '/content/ultralytics/runs/detect/YOLO11m_VisDrone/run/weights/best.pt'

# Execute the YOLO validation command with the corrected model path
!yolo val model={model_path} data=dataset.yaml save=True

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,037,742 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 39.1±19.3 MB/s, size: 97.9 KB)
val: Scanning /root/.cache/kagglehub/datasets/banuprasadb/visdrone-dataset/versions/1/VisDrone_Dataset/VisDrone2019-DET-val/labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 114.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 2.0it/s 17.9s
                   all        548      38759       0.56      0.446      0.464      0.284
            pedestrian        520       8844      0.603       0.49      0.531      0.254
                people        482       5125      0.613      0.351      0.398      0.159
               bicycle        364       1287      0.327      0.228      0.216        0.1
                   car        515      14064    

## Final Task

### Subtask:
Confirm that the metrics are displayed correctly and the zipped archive containing all model artifacts is available for download.


## Summary:

### Q&A
Yes, the evaluation metrics were displayed correctly, and a zipped archive containing all model artifacts is available for download.

### Data Analysis Key Findings
*   The final evaluation metrics for the trained model are:
    *   mAP@0.5: 0.4627
    *   mAP@0.5:0.95: 0.2835
    *   Precision: 0.5625
    *   Recall: 0.4430
*   A zip archive named `YOLO11m_VisDrone_run_artifacts.zip` containing all training artifacts from `/content/ultralytics/runs/detect/YOLO11m_VisDrone/run` was successfully created and made available for download.
*   The YOLO validation command executed successfully with the corrected absolute model path, providing overall validation metrics: Box Precision (P) of 0.56, Box Recall (R) of 0.446, Box mAP50 of 0.464, and Box mAP50-95 of 0.284.

### Insights or Next Steps
*   The reported metrics provide a baseline understanding of the model's performance on the VisDrone dataset, which can be used for further optimization or comparison.
*   The availability of a comprehensive zip archive ensures that all training artifacts are preserved, allowing for reproducibility, detailed post-analysis, or deployment of the trained model.
